# AutoTrain for Non-LLM Tasks: Text & Image Classification

AutoTrain Advanced is not just for LLMs. The same tool covers a wide range
of tasks via different subcommands:

| Task | CLI subcommand | Dataset format |
|------|---------------|----------------|
| LLM SFT | `autotrain llm` | CSV with `text` column |
| **Text classification** | `autotrain text-classification` | CSV with `text` + `target` columns |
| **Image classification** | `autotrain image-classification` | ZIP with class subfolders |
| Object detection | `autotrain object-detection` | COCO-format JSON + images |
| Token classification | `autotrain token-classification` | CSV with `tokens` + `tags` |
| Tabular | `autotrain tabular` | CSV with feature cols + `target` |

This notebook covers **text classification** (Section A) and
**image classification** (Section B).

## Environment Setup

```bash
cd projects/autotrain_ui
uv sync --no-install-workspace
bash setup.sh
uv run ipython kernel install --user --env VIRTUAL_ENV ../../.venv --name=autotrain_ui
```

Select the **`autotrain_ui`** kernel (Python 3.10.x required).

In [ ]:
import sys, subprocess, os, pathlib, csv
assert sys.version_info[:2] == (3, 10), f'Need Python 3.10.x, got {sys.version_info[:2]}'
import autotrain
print(f'Python {sys.version}')
print(f'AutoTrain {autotrain.__version__}')

# Discover available subcommands
r = subprocess.run(['autotrain', '--help'], capture_output=True, text=True)
print(r.stdout[:2000])

In [ ]:
import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path('../../../').resolve()))
from dotenv_config import dot_env_settings

if not dot_env_settings.HF_TOKEN:
    raise EnvironmentError('HF_TOKEN not set — see .env.example in repo root')

os.environ['HF_TOKEN'] = dot_env_settings.HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = dot_env_settings.HF_TOKEN
print('HF_TOKEN set.')

---
## Section A — Text Classification

**Task:** multi-class topic classification (3 categories: science, sports, politics)

AutoTrain trains an encoder model (BERT-family by default) with a
classification head. No LoRA or quantisation is needed — full fine-tuning
on a small encoder is fast.

### A1. Dataset Format

> **Gotcha #1 — Text classification needs `target`, not `label`.**
> The column name requirements differ by task:
> - LLM SFT: `text` (one column)
> - Text classification: `text` + `target`
>
> Using `label` instead of `target` causes AutoTrain to fail at
> dataset loading with a confusing KeyError.

> **Gotcha #2 — `target` can be an integer or a string.**
> AutoTrain encodes string labels automatically. Both `target=0` and
> `target='science'` work. Integers train slightly faster since no encoding
> step is needed.

In [ ]:
text_examples = [
    # Science
    {'text': 'Researchers discover a new exoplanet in the habitable zone.', 'target': 'science'},
    {'text': 'CRISPR gene editing shows promise for treating hereditary diseases.', 'target': 'science'},
    {'text': 'The James Webb Telescope captures images of the early universe.', 'target': 'science'},
    {'text': 'Scientists synthesise a new material with record-breaking conductivity.', 'target': 'science'},
    {'text': 'Quantum computing milestone: 1,000-qubit processor demonstrated.', 'target': 'science'},
    # Sports
    {'text': 'The home team wins the championship in an overtime thriller.', 'target': 'sports'},
    {'text': 'Transfer window: striker moves to rival club for record fee.', 'target': 'sports'},
    {'text': 'Olympic gold medallist breaks the 100m world record.', 'target': 'sports'},
    {'text': 'Tennis star retires after 20 years and 23 Grand Slam titles.', 'target': 'sports'},
    {'text': 'League table update: three teams level on points at the top.', 'target': 'sports'},
    # Politics
    {'text': 'Parliament passes new climate legislation with bipartisan support.', 'target': 'politics'},
    {'text': 'Prime Minister announces early general election for next month.', 'target': 'politics'},
    {'text': 'Trade negotiations between the two nations stall over tariffs.', 'target': 'politics'},
    {'text': 'Opposition party elects new leader after leadership contest.', 'target': 'politics'},
    {'text': 'Summit communiqué calls for renewed commitment to multilateralism.', 'target': 'politics'},
]

text_data_dir = pathlib.Path('./autotrain_text_cls_data')
text_data_dir.mkdir(exist_ok=True)
train_csv = text_data_dir / 'train.csv'

with open(train_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['text', 'target'])
    writer.writeheader()
    writer.writerows(text_examples)

print(f'Wrote {len(text_examples)} examples to {train_csv}')
import pandas as pd
print(pd.read_csv(train_csv).head())

### A2. Explore Available Flags


In [ ]:
r = subprocess.run(['autotrain', 'text-classification', '--help'],
                   capture_output=True, text=True)
print(r.stdout[:3000])

### A3. Run Text Classification

> **Gotcha #3 — Validation split is carved out automatically.**
> If you only provide `train.csv`, AutoTrain creates a validation split
> internally. To use your own validation set, add `valid.csv` to the
> same data directory.

In [ ]:
TEXT_CLS_PROJECT = 'autotrain_topic_clf'
TEXT_MODEL = 'distilbert-base-uncased'

cmd_text_cls = [
    'autotrain', 'text-classification',
    '--model', TEXT_MODEL,
    '--data-path', str(text_data_dir.resolve()),
    '--train-split', 'train',
    '--project-name', TEXT_CLS_PROJECT,
    '--text-column', 'text',
    '--target-column', 'target',   # note: 'target', not 'label'
    '--lr', '2e-5',
    '--batch-size', '4',
    '--epochs', '5',
]

print('Command:')
print('  ' + ' '.join(cmd_text_cls))
print()

process = subprocess.Popen(
    cmd_text_cls,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Text classification training failed (exit {process.returncode})')
print('Done.')

In [ ]:
# Inspect output
out_dir = pathlib.Path(TEXT_CLS_PROJECT)
if out_dir.exists():
    for f in sorted(out_dir.rglob('*')):
        if f.is_file():
            print(f'  {f.relative_to(out_dir)}  ({f.stat().st_size:,} bytes)')
else:
    print('Output directory not found — did training complete?')

---
## Section B — Image Classification

**Task:** 3-class image classification (red / green / blue synthetic images)

AutoTrain expects images in a ZIP archive with one subfolder per class.
We create a small synthetic dataset using NumPy and PIL to keep this
notebook self-contained. In production, substitute your own images.

### B1. Dataset Format

> **Gotcha #4 — AutoTrain image classification requires a ZIP, not a CSV.**
> Unlike text tasks (CSV), image classification expects:
> ```
> dataset.zip/
> ├── train/
> │   ├── class_a/  image1.jpg  image2.jpg ...
> │   ├── class_b/  ...
> │   └── class_c/  ...
> └── valid/         (optional; carved automatically if absent)
>     └── ...
> ```
> The class names are inferred from the subfolder names.

> **Gotcha #5 — Minimum ~5 images per class.**
> AutoTrain's internal validation split may leave fewer than 2 images
> per class if the dataset is too small, causing a crash. Use at least
> 10–20 real images per class for reliable results.

In [ ]:
import numpy as np
from PIL import Image
import zipfile, io

CLASSES = ['red', 'green', 'blue']
IMAGES_PER_CLASS = 20   # synthetic images per class
IMG_SIZE = 64            # 64x64 pixels

def make_synthetic_image(dominant_channel: int, noise: float = 0.3) -> Image.Image:
    """Create an IMG_SIZE x IMG_SIZE RGB image dominated by one colour channel."""
    arr = np.random.uniform(0, noise, (IMG_SIZE, IMG_SIZE, 3))
    arr[:, :, dominant_channel] = np.random.uniform(0.7, 1.0, (IMG_SIZE, IMG_SIZE))
    return Image.fromarray((arr * 255).astype(np.uint8))

zip_path = pathlib.Path('./autotrain_image_data.zip')
with zipfile.ZipFile(zip_path, 'w') as zf:
    for class_idx, class_name in enumerate(CLASSES):
        for i in range(IMAGES_PER_CLASS):
            img = make_synthetic_image(class_idx)
            buf = io.BytesIO()
            img.save(buf, format='JPEG')
            zf.writestr(f'train/{class_name}/{class_name}_{i:03d}.jpg', buf.getvalue())

print(f'Created {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)')
print(f'Contents: {CLASSES}, {IMAGES_PER_CLASS} images each')

### B2. Explore Image Classification Flags


In [ ]:
r = subprocess.run(['autotrain', 'image-classification', '--help'],
                   capture_output=True, text=True)
print(r.stdout[:3000])

### B3. Run Image Classification

> **Gotcha #6 — The `--data-path` argument points to the ZIP file, not a directory.**
> For image classification, pass the path to the `.zip` file directly.
> AutoTrain unpacks it internally.

In [ ]:
IMG_CLS_PROJECT = 'autotrain_colour_clf'
IMG_MODEL = 'google/vit-base-patch16-224'

cmd_img_cls = [
    'autotrain', 'image-classification',
    '--model', IMG_MODEL,
    '--data-path', str(zip_path.resolve()),   # path to the ZIP file
    '--project-name', IMG_CLS_PROJECT,
    '--lr', '2e-5',
    '--batch-size', '8',
    '--epochs', '3',
]

print('Command:')
print('  ' + ' '.join(cmd_img_cls))
print()

process = subprocess.Popen(
    cmd_img_cls,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Image classification training failed (exit {process.returncode})')
print('Done.')

In [ ]:
out_dir_img = pathlib.Path(IMG_CLS_PROJECT)
if out_dir_img.exists():
    for f in sorted(out_dir_img.rglob('*')):
        if f.is_file():
            print(f'  {f.relative_to(out_dir_img)}  ({f.stat().st_size:,} bytes)')
else:
    print('Output directory not found — did training complete?')

## Summary

### Gotcha recap

| # | Gotcha | Fix |
|---|--------|-----|
| 1 | Text cls needs `target`, not `label` | Column must be named `target` |
| 2 | `target` type | Integers or strings both work |
| 3 | No explicit val split needed | AutoTrain carves one automatically |
| 4 | Image cls needs ZIP, not CSV | Structure: `zip/train/class_name/*.jpg` |
| 5 | Too few images per class | Use ≥10–20 images per class |
| 6 | `--data-path` for images is the ZIP path | Pass the `.zip` file, not a directory |

### Task vs. dataset format quick reference

| Task | `--data-path` points to | Required columns / structure |
|------|------------------------|------------------------------|
| `llm` | directory with CSV | `text` |
| `text-classification` | directory with CSV | `text`, `target` |
| `image-classification` | ZIP file | `train/<class>/*.jpg` |
| `token-classification` | directory with CSV | `tokens`, `tags` |
| `tabular` | directory with CSV | feature cols + `target` |

**Next steps:** replace the synthetic colour images with a real dataset
(e.g., download `beans` from HF and repackage as a ZIP), try
`autotrain tabular` on a CSV regression task, or add `--push-to-hub`
to publish the trained model automatically.